## 💢 Linguistic markers of subtle discrimination among mental health professionals*

Organizes iterative, independent, co-annotation of audit correspondence field experiment responses received from mental health professionals. Samples pilot + parent study data by co-annotation cycle, computes Cohen's $\kappa$, flags discrepant tagging decisions for in-person / LLM-assisted deliberation. Compiles $\mathcal{d}$<sub>annotated</sub> ($n$ = 1,923) for `BERTForSequenceClassification` train-test cycles.

**Pre-analysis plan filed Sep 20, 2025 on Open Science Framework: [doi: 10.17605/OSF.IO/WGU8Q](https://doi.org/10.17605/OSF.IO/WGU8Q).**

*Provided in the spirit of open science. This notebook is not integral to the reproducible analytic pipeline. Annotated data subsets used for schema refinement and inter-annotater agreement tests are available (sanitized) upon reasonable request. 

🔩 `annotation_pipeline.ipynb`<br>
Simone J. Skeen, Daniel A. Lau x Claude Code (05-25-2026)

1. [Prepare](#1-prepare)
2. [Sample + Triangulate](#2-sample--triangulate)
3. [Compile + Tag $\mathcal{d}$<sub>annotated</sub>](#3-compile--tag-annotated)

### 1. Prepare
Installs, imports, and downloads requisite models and packages.
***

**Install**

In [ ]:
%%capture

%pip install openai

**Import**

In [ ]:
# Standard library
import os
import sys
import warnings

# Third-party
from contextlib import chdir
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from sklearn.metrics import cohen_kappa_score
from sklearn.utils import shuffle

# Local
from annotate import (
    calculate_kappa_by_cycle,
    condense_response_frame,
    ner_redact_response_texts,
    remove_index_artifacts,
    sample_by_cycle,
    )

# Download spaCy model
spacy.cli.download('en_core_web_lg')

# Output preferences
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

#### _Annotation schema_
**Table i. MHP response quality subconstructs: working definitions**

|Subconstruct|Alias|Definition|
|---|---|---|
|affirm|`afrm`|Responding in a manner that explicitly reinforces prospective client’s decision to seek therapy|
|agent|`agnt`|Interfacing with a prospective client via an automatic message or non-MHP staffer such as a scheduler or office coordinator|
|burden†|`brdn`|Interjections and descriptions of administrative burden as reality of provisioning social services|
|demand†|`dmnd`|Interjections and descriptions of administrative burden that use _imperative semantics_|
|fit|`fitt`|Explicitly mentioningthe importance of client-MHP rapport, and/or alliance in a client-empowering manner|
|justify‡|`just`|Providing a logistical reason for being unable to meet with prospective clientas a personal/professional courtesy. Reasons may include full caseload, geographic bounds of licensure, imminent retirement, etc.|
|rebound†|`rbnd`|Queries imposed on the prospective client in response to appointment-seeking inquiries|
|reflect|`refl`|Echoing, restating, and/or reusing the prospective client’s own words or description of their presenting problem|

†To be examined via reflexive thematic analysis, given sparsity; ‡collapsed with "Rejection Reasoning" parent study outcomes coding. The complete [Phase 1 & Phase 2 open annotation schemas](https://osf.io/8ad2t/overview), including synthetic examples and edge case rulesets developed inductively by SJS and DAL, are available on the Open Science Framework. Few-shot GPT classification prompts are specified in `annotation_schema.YAML`



In [ ]:
%%script false --no-raise-error

# Map subdirectory structure
.
├── src
├── data
└── annotation

In [ ]:
# Set working directory to project root; define data paths
if os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')

# Add `src` subdirectory to import path for `annotate.py` custom modules 
sys.path.insert(0, 'src')

DATA_DIR = 'data'    ### subdirectory to receive data from parent study team
ANNOTATION_DIR = 'annotation'   ### subdirectory for annotation tasks + data derivatives

### 2. Sample + Triangulate
Randomly samples cycle-specific MHP response subsets for annotation. Computes Cohen's $\kappa$, dummy codes discrepant tags for in-person deliberation.
***
**Data dictionary**<br>
`d_pilot_n755`: complete valid deduplicated MHP response data ($n$ = 755) collected in the course of pilot audit correspondence study: [Fumarco et al. _Am J Health Econ_. 2024;10:182–214. doi: 10.1086/728931](https://pmc.ncbi.nlm.nih.gov/articles/PMC12547971/).<br>
`d_parent_250`: initial valid deduplicated MHP response data ($n$ = 250) collected following launch of parent study. Pre-analysis plan (filed January 28, 2026 on American Economic Association RCT Registery) includes detailing of data-generating procedures: [RCT ID: AEARCTR-0016309.](https://www.socialscienceregistry.org/trials/16309)<br>

**Narrative summary**<br>
Randomly sampled subsets ($n$ = 80–100) of pilot, and eventually pilot + parent study data, were independently annotated by human annotators SJS & DAL, until preregistered Cohen's $\kappa$ benchmarks > 0.75 were achieved over all tags with occurences frequent enough to train clasification models.

In [ ]:
### TODO 5/28: Replace narrative summaries w/ preprint excerpts

# Load pilot data
d = pd.read_excel(f'{ANNOTATION_DIR}/d_pilot_n755.xlsx', index_col=0)
d.head(3)

#### _Cycle 0: $\mathcal{d}$<sub>pilot</sub> data ($n$ = 755) exclusive_

In [ ]:
# Sample and prepare cycle 0

### NOTE: Predated development of `sample_by_cycle` helper fx called in subsequent cycles

d_cycle_0 = d.sample(n=50, random_state=56)
d_cycle_0.reset_index(drop=True, inplace=True)

tag_cols = ['afrm', 'agnt', 'fitt', 'just', 
            'prbl', 'refl', 'rtnl', 'note']
d_cycle_0[tag_cols] = ' '

drop_cols = ['EmailPairID', 'WithinPatientID', 'FirstInPair', 
             'pilot', 'MHP ID#']
d_cycle_0.drop(columns=drop_cols, 
               axis=1, 
               inplace=True,
               )

d_cycle_0.head(3)
d_cycle_0.to_excel(f'{ANNOTATION_DIR}/d_cycle_0.xlsx', index=True)

In [ ]:
# Post-annotation IAA
with chdir(ANNOTATION_DIR): d, kappa_results = calculate_kappa_by_cycle(0)

#### _Cycle 1: $\mathcal{d}$<sub>pilot</sub> data ($n$ =755) exclusive_

In [ ]:
d_cycle_1 = sample_by_cycle(
    d, 
    sample_size=80, 
    cycle_number=1
    )

d_cycle_1.info()
d_cycle_1.head(3)

In [ ]:
# Post-annotation IAA
with chdir(ANNOTATION_DIR): d, kappa_results = calculate_kappa_by_cycle(1)

#### Cycle 2: $\mathcal{d}$<sub>pilot</sub> data ($n$ = 755) exclusive

In [ ]:
d_cycle_2 = sample_by_cycle(
    d, 
    sample_size=80, 
    cycle_number=2,
    )

d_cycle_2.info()
d_cycle_2.head(3)

In [ ]:
# Post-annotation IAA
with chdir(ANNOTATION_DIR): d, kappa_results = calculate_kappa_by_cycle(2)

#### _Cycle 3: $\mathcal{d}$<sub>pilot</sub> (n = 755) and $\mathcal{D}$<sub>parent</sub> (n  =250) data_

In [ ]:
# Import available pilot and parent study data
d_pilot_n755 = pd.read_excel(
    f'{ANNOTATION_DIR}/d_pilot_n755.xlsx',
    index_col = 'index',
    )

d_parent_n250 = pd.read_excel(
    f'{ANNOTATION_DIR}/d_parent_n250.xlsx',
    index_col = [0],
    )

# De-identify
d_parent_n250['text'] = d_parent_n250['text'].astype(str).apply(lambda i: ner_redact_response_texts(i))

# Encode `pilot` var to disambiguate
d_pilot_n755['pilot'] = 1

In [ ]:
# Concatenate df & shuffle response order
d_append = pd.concat([
    d_pilot_n755,
    d_parent_n250,
    ])

d_append = shuffle(
    d_append,
    random_state = 56,
    )

In [ ]:
d_cycle_3 = sample_by_cycle(
    d_append,
    100,  ### sample_size = 80
    3,    ### cycle_number = 3
    )

d_cycle_3.info()
d_cycle_3.head(3)

In [ ]:
# Post-annotation IAA
with chdir(ANNOTATION_DIR): d, kappa_results = calculate_kappa_by_cycle(3)

### NOTE: Preregistered Kappa > 0.75 obtained in Cycle 3

**Viz.**

In [ ]:
# Initialize IAA kappa lists
iaa={
    'cycle': [0, 1, 2, 3],
    'afrm_k': [0.66, 0.79, 0.55, 0.88], 'agnt_k': [0.65, 0.39, 0.56, 0.85], 
    'fitt_k': [0.00, 0.65, 0.93, 0.78], 'just_k': [0.46, 0.69, 0.64, 0.68],
    'rbnd_k': [0.22, 0.00, 0.36, 0.80], 'refl_k': [0.76, 0.79, 0.89, 0.78],
    }

# Create df
d_iaa=pd.DataFrame(iaa)

In [ ]:
# Plot kappa * cycle * tag
sns.set(style='whitegrid', rc=None)

# Custom x-axis labels
custom_labels=['cycle 0', 'cycle 1',
               'cycle 2', 'cycle 3']

# Plot
plt.figure(figsize = (8, 5.5))

# Jitter settings
jitter_amount = 0.08
np.random.seed(42)  # For reproducibility

plt.scatter(
    d_iaa['cycle'] + np.random.uniform(-jitter_amount, jitter_amount, len(d_iaa)),
    d_iaa['afrm_k'], label="afrm",
    marker='o', alpha=0.9, color='#daf7a1', s=60,
    )

plt.scatter(
    d_iaa['cycle'] + np.random.uniform(-jitter_amount, jitter_amount, len(d_iaa)),
    d_iaa['agnt_k'], label="agnt",
    marker='o', alpha=0.6, color='#ffc000', s=60,
    )

plt.scatter(
    d_iaa['cycle'] + np.random.uniform(-jitter_amount, jitter_amount, len(d_iaa)),
    d_iaa['fitt_k'], label="fitt",
    marker='o', alpha=0.9, color='#FFCEE0', s=60,
    )

plt.scatter(
    d_iaa['cycle'] + np.random.uniform(-jitter_amount, jitter_amount, len(d_iaa)),
    d_iaa['just_k'], label="just",
    marker='o', alpha=0.6, color='#6ED8FA', s=60,
    )

plt.scatter(
    d_iaa['cycle'] + np.random.uniform(-jitter_amount, jitter_amount, len(d_iaa)),
    d_iaa['rbnd_k'], label="rbnd",
    marker='o', alpha=0.6, color='#b485bc', s=60,
    )

plt.scatter(
    d_iaa['cycle'] + np.random.uniform(-jitter_amount, jitter_amount, len(d_iaa)),
    d_iaa['refl_k'], label="refl",
    marker='o', alpha=0.6, color='#FF9D48', s=60,
    )

# Define axis labels & title
plt.title("Fig. i: Cohen's $\kappa$ >0.75 achieved over $k$ = 3 annotation cycles.")
#plt.xlabel("Annotation cycle")
plt.ylabel("Cohen's $\kappa$")

ax=plt.gca()
ax.set_yticks([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 
               0.6, 0.7, 0.8, 0.9, 1.0])
ax.tick_params(axis='y', labelsize=11)

# Set horizontal gridlines
plt.grid(axis='x')

# Interpolate 0.75 IAA benchmark (--)
plt.axhline(
    y=0.75,
    color='red',
    linewidth=0.6,
    linestyle='--',
    )

plt.text(-0.12, 0.74, "$\kappa$ = 0.75 benchmark", color='red', va='top', fontsize=9.5)

# Fix y-axis at 0
plt.ylim(0, None)

# Format legend
plt.legend(
    loc='lower right',
    ncol=3,
    frameon=True,
    edgecolor='#e1dada',
    fontsize=9.5,
    )

# Despine
sns.despine(right=True)

# Insert x-axis ticks
plt.xticks(
    ticks=[0, 1, 2, 3],
    labels=['cycle 0', 'cycle 1', 
            'cycle 2', 'cycle 3'],
    rotation=45,
    ha='right',
    fontsize=11,
    )

# Degrid & tighten
plt.grid(False)
plt.tight_layout()

# Save & display
#plt.savefig(f'iaa_scatter.png', dpi = 300)
plt.figtext(
    0.12, 
    -0.02, 
    "Annotation definitions and handling of edge cases were refined via human-human deliberation between cycles. \nCycle 0 was motivated primarily to pilot and stress test procedures.", 
    ha='left', 
    fontsize=9.5, 
    style='italic',
    )

plt.show();

### 3. Compile + Tag $\mathcal{d}$<sub>annotated</sub>
Structures and sequentially merges traint-test set $\mathcal{d}$<sub>annotated</sub> ($n$ = 1,923)
***
**Narrative summary**<br>
With Cohen's $\kappa$ >0.75 obtained on `afrm`, `agnt`, `fitt`, `rbnd`, and `refl` tags in co-annotation + triangulation Cycle 3, all pilot MHP response data ($\mathcal{d}$<sub>pilot</sub>, $n$ = 755) and parent MHP response data ($\mathcal{D}$<sub>parent</sub>) collected as of November 5, 2024 ($n$ = 250) were compiled for tagging by a single annotator: DAL.

In [ ]:
# Load and prepare d_pilot_n755
d = pd.read_excel(f'{ANNOTATION_DIR}/d_pilot_n755.xlsx', index_col='index')
d['pilot'] = 1
d['MHP ID'] = '.'
d['Unique ID'] = '.'

d_pilot_n755 = d[[
    'EmailPairID', 'WithinPatientID', 'FirstInPair',
    'MHP ID', 'Unique ID', 'pilot', 'text',
]]

d_pilot_n755.to_excel(f'{ANNOTATION_DIR}/d_pilot_n755.xlsx', index=True)

In [ ]:
# Load and prepare d_parent_250
d = pd.read_excel(
  f'{DATA_DIR}/Response Spreadsheet for Simone_01.xlsx', ### initial parent study batch from BAL (11/5/2024)
  index_col=[0],
  ) 

# Call `condense_response_frame` helper fx
d_parent_n250 = condense_response_frame(d)

d_parent_n250.info()
d_parent_n250.head(3)

# Anonymize
d_parent_n250['text'] = d_parent_n250['text'].astype(str).apply(lambda i: ner_redact_response_texts(i))

d_parent_n250.to_excel(f'{ANNOTATION_DIR}/d_parent_n250.xlsx', index=True)

#### 3.1 `d_annotated_01_n1005` (755 + 250): DAL annotation set
_The initial training dataset single-annotated by DAL: $n$ = 1-900._

In [ ]:
# Concatenate
d_annotated_01_n1005 = pd.concat([
  d_pilot_n755,
  d_parent_n250,
  ])

# Assign annotation columns
d_annotated_01_n1005 = d_annotated_01_n1005.assign(
    prbl = ' ', agnt = ' ', afrm = ' ',
    brdn = ' ', dmnd = ' ', fitt = ' ',
    just = ' ', rbnd = ' ', refl = ' ',
    rtnl = ' ', note = ' ',
    )

# Shuffle
d_annotated_01_n1005 = shuffle(
  d_annotated_01_n1005,
  random_state = 56,
  )

In [ ]:
# Reset idx
d_annotated_01_n1005.reset_index(
    inplace = True,
    drop = True,
    )

#### _DAL annotates 1–900 (of 1,005) starting 1/16/2025_
DAL single-annotated $n$ = 900 training instances. From this initial training dataset, a subset of $n$ = 100 was randomly sampled, and dual-annotated, to assure IAA fidelity post-triangulation.

In [ ]:
# Inspect & save
d_annotated_01_n1005.info()
d_annotated_01_n1005.head(3)
d_annotated_01_n1005.to_excel(f'{ANNOTATION_DIR}/d_annotated_01_n1005.xlsx', index = True)

#### 3.2. `d_iaa_dal` / `d_iaa_sjs` dual-annotated IAA subset
_From these 900, an $n$ = 100 subset was sampled for dual annotation + post-triangulation IAA testing to ensure IAA fidelity in training data._

In [ ]:
d_iaa_prelim = d.iloc[:899].copy()

In [ ]:
# Sample n = 100, already annotated by DAL
d_iaa_dal = d_iaa_prelim.sample(
    n = 100,
    random_state = 42,
    ).copy()

# Inspect & save
d_iaa_dal.info()
d_iaa_dal.to_excel(f'{ANNOTATION_DIR}/d_iaa_dal.xlsx', index=True)

In [ ]:
# Duplicate DAL-annotated sample & overwrite tags with blank cells
d_iaa_sjs = d_iaa_dal.copy()
d_iaa_sjs[[
    'prbl', 'agnt', 'afrm', 'brdn',
    'dmnd', 'fitt', 'just', 'rbnd',
    'refl', 'rtnl', 'note',
    ]] = ''

# Save
d_iaa_sjs.to_excel(f'{ANNOTATION_DIR}/d_iaa_sjs.xlsx', index=True)

#### _SJS dual-annotates DAL-annotated 100 (of 900)_

In [ ]:
# Run `calculate_kappa_by_cycle` helper fx (10/6/2025) / training IAA = "Cycle 4"
d, kappa_results = calculate_kappa_by_cycle(4)

### NOTE: "Excellent" Kappa > 0.75 sustained in training dataset, excepting `afrm` (0.66) 7 `agnt` (n/% insufficient)

#### 3.3. `d_annotated_02_n1923` (1005 + 918): SJS annotation set
_The complete (pre-augmentation) training dataset, partially single-annotated ($n$ $\leq$ 900) by DAL._<br>

In [ ]:
d = pd.read_excel(
    f'{DATA_DIR}/Response Spreadsheet for Simone_02.xlsx', ### initial parent study batch from BAL (11/5/2024)
    )

# Assign idx
d.index.name = 'index'

# Call `condense_response_frame` helper fx
d_parent_n918 = condense_response_frame(d)

# Assign annotation columns
d_parent_n918 = d_parent_n918.assign(
    prbl = ' ', agnt = ' ', afrm = ' ',
    brdn = ' ', dmnd = ' ', fitt = ' ',
    just = ' ', rbnd = ' ', refl = ' ',
    rtnl = ' ', note = ' ',
    )

# Inspect & save
d_parent_n918.info()
d_parent_n918.head(3)
d_parent_n918.to_excel(f'{ANNOTATION_DIR}/d_parent_n918.xlsx', index=True)

In [ ]:
d_annotated_01_dal_n1005 = pd.read_excel(f'{ANNOTATION_DIR}/d_annotated_01_dal_n1005.xlsx', index_col=[0])

# Eliminate artifacts
if d_annotated_01_dal_n1005.index.name=='/;':
    d_annotated_01_dal_n1005.reset_index(
        inplace = True,
        )

if '/;' in d_annotated_01_dal_n1005.columns:
    d_annotated_01_dal_n1005.drop(
        columns = ['/;'],
        inplace = True,
        )

# Assign idx & inspect
d_annotated_01_dal_n1005.index.name='index'
d_annotated_01_dal_n1005.head(3)

In [ ]:
# Concatenate
d_annotated_02_n1923 = pd.concat([
  d_annotated_01_dal_n1005,
  d_parent_n918,
  ])

#### _SJS annotates 901–1,923 (of 1,923) starting 7/17/2025_
With an additional $n$ = 918 parent study MHP responses received by March 10, 2025, these data were appended to the initial training dataset to create the complete $\mathcal{d}$<sub>annotated</sub> ($n$ = 1,923). The balance of training instances ($n$ $\gt$ 900) were then single-annotated by SJS.

In [ ]:
# Reset idx
d_annotated_02_n1923.reset_index(
    inplace = True,
    drop = True,
    )

# Inspect & save
d_annotated_02_n1923.info()
d_annotated_02_n1923.head(3)
d_annotated_02_n1923.to_excel(f'{ANNOTATION_DIR}/d_annotated_02_n1923.xlsx', index=True)

#### _SJS dual-annotates DAL-annotated 100§ (of 1,923)_
§Due to `agnt` instances' sparsity, $\mathcal{d}$<sub>annotated</sub> DAL `agnt` tags were pulled via disproportianate stratified sampling (4.47% `agnt` = 1) and dual-annotated by SJS to compute a valid IAA.

In [ ]:
d = pd.read_excel(f'{ANNOTATION_DIR}/d_annotated_02_dal_sjs_n1923.xlsx', index_col=[0])

# Housekeeping: int64 convert, fillna
targets=[
  'prbl', 'agnt', 'afrm', 'brdn',
  'dmnd', 'fitt', 'just', 'rbnd',
  'refl',
  ]

d[targets]=(
    d[targets]
    .replace(r'^\s*$', 0, regex=True,)
    .fillna(0)
    .astype('int64')
    )

d.info()
d.head(3)

# Generate descriptives: % tags = 1 in complete annotated `d_annotated_n1923` df
t_pct=d[targets].mean() * 100
print("\n --- target % --- \n")
print(t_pct)

### NOTE: 4.47% `agnt` = 1

In [ ]:
d = pd.read_excel(f'{ANNOTATION_DIR}/d_annotated_01_dal_n1005.xlsx', index_col=[0])

# Eliminate artifact
if d.index.name=='/;':
    d.reset_index(inplace=True)

if '/;' in d.columns:
    d.drop(columns=['/;'], inplace=True)

# Housekeeping: int64 convert, fillna
targets = [
  'prbl', 'agnt', 'afrm', 'brdn',
  'dmnd', 'fitt', 'just', 'rbnd',
  'refl',
  ]

d[targets]=(
    d[targets]
    .replace(r'^\s*$', 0, regex=True,)
    .fillna(0)
    .astype('int64')
    )

d.info()
d.head(3)

In [ ]:
# Disproportianate stratified sampling: 5 [4.47% 'agnt' = 1] vs. 95 split
seed = 56

sample_agnt1=d[d['agnt']==1].sample(n=5, random_state=seed)
sample_agnt0=d[d['agnt']==0].sample(n=95, random_state=seed)

# (Re)concatenate
d_agnt_dal_iaa=pd.concat([sample_agnt1, sample_agnt0]).sample(frac=1, random_state=seed).reset_index(drop=True)

# Inspect & verify
d_agnt_dal_iaa.info()
pct_agnt=d_agnt_dal_iaa['agnt'].mean() * 100
print("% agent = 1", pct_agnt)

In [ ]:
# Shuffle
d_agnt_dal_iaa = shuffle(d_agnt_dal_iaa, random_state = seed).reset_index(drop = True)

# Reduce to `agnt` & `rtnl` cols
cols_to_drop=[
    'prbl', 'afrm', 'brdn', 'dmnd', 'fitt',
    'just', 'rbnd', 'refl', 'note',
    ]

d_agnt_dal_iaa=d_agnt_dal_iaa.drop(columns=cols_to_drop, errors='ignore')

# Delete DAL tags for SJS annotation set
d_agnt_sjs_iaa=d_agnt_dal_iaa.copy()
d_agnt_sjs_iaa['agnt']=' '
d_agnt_sjs_iaa['rtnl']=' '

# Inspect & verify
d_agnt_dal_iaa.info()
d_agnt_dal_iaa.head(1)

In [ ]:
# Save
d_agnt_dal_iaa.to_excel(f'{ANNOTATION_DIR}/d_agnt_dal_iaa.xlsx', index=True)
d_agnt_sjs_iaa.to_excel(f'{ANNOTATION_DIR}/d_agnt_sjs_iaa.xlsx', index=True)

In [ ]:
# Reload after SJS (masked) independent annotation
d_agnt_dal_iaa = pd.read_excel(f'{ANNOTATION_DIR}/d_agnt_dal_iaa.xlsx', index_col=[0])
d_agnt_sjs_iaa = pd.read_excel(f'{ANNOTATION_DIR}/d_agnt_sjs_iaa.xlsx', index_col=[0])

In [ ]:
# Disambiguate & merge
d_agnt_dal_iaa = d_agnt_dal_iaa.rename(columns={'agnt': 'agnt_dal'})
d_agnt_sjs_iaa = d_agnt_sjs_iaa.rename(columns={'agnt': 'agnt_sjs'})

d_agnt_iaa = pd.merge(
    d_agnt_dal_iaa,
    d_agnt_sjs_iaa,
    left_index = True,
    right_index = True,
    )

In [ ]:
# Assign IAA cols
cols = ['agnt_dal', 'agnt_sjs']

# Replace blank/whitespace and NaN with 0, convert to int
d_agnt_iaa[cols] = (
    d_agnt_iaa[cols]
    .replace(r'^\s*$', 0, regex = True)
    .fillna(0)
    .astype('int64')
    )

# (Re)run IAA
kappa = cohen_kappa_score(d_agnt_iaa['agnt_dal'], d_agnt_iaa['agnt_sjs'])
print("Cohen's Kappa:", kappa)

### NOTE: `agnt` Kappa = 0.50 after stratified sampling; `afrm` & `agnt` require LLM-assisted negotatiated agreement

End of `annotation_pipeline.ipynb`